# LS-NEQ 01_baseline_and_directions — Baselines und Richtungen

**Dossier-Stufe B** (`lsne.experiments.STAGES["B"]`, siehe `docs/DESIGN.md` §8).
Dieses Notebook wird von `scripts/build_notebooks.py` erzeugt; Änderungen bitte dort
vornehmen und neu generieren (`python scripts/build_notebooks.py`).

**Zweck.** Misst für jede Variante die unveränderte Baseline (log p(Ziel), Entropie, Masse auf den 100 häufigsten Tokens) und baut die Richtungsbank an der Dossier-Stelle (Block-Eingang des letzten Blocks, letzte Kontextposition): `ent` (∂H/∂h, State-2143-Konstruktion), `lang_dom` (Difference-of-Means D−E, denoised), `lang_probe`, `freq` (Häufigkeitsachse), 32 Zufallsrichtungen, optional SAE-Feature-Richtungen. Geometriebericht: Kosinusmatrix, Unembedding-Projektion (beförderte/unterdrückte Tokens), P_freq vs. Entropie.

**Eingaben.** Stimulusbank (Stufe A wird bei Bedarf intern neu aufgebaut), Anker, Unigramm-Zählung (Range-Requests, gecacht).

**Ausgaben.** Tabellen `baseline`, `direction_meta`, `cosines`, `unembed`, `pfreq_entropy_tokens`, `pfreq_entropy_by_class` [+ `sae_features`, `sae_summary`]; Richtungsbank `directions.npz` + `directions_meta.json` (von den Stufen C–E wiederverwendet).
Alle Tabellen liegen als CSV (+ Parquet) im Laufverzeichnis
`<RUNS_ROOT>/B/<modell>@<revision>/` zusammen mit `manifest.json`
(Git-SHA, Modell, dtype, Hook, Anker-Hashes, Zeitstempel); Abbildungen unter
`<OUT_ROOT>/figures/01_baseline_and_directions/`; am Ende eine kurze `summary_01_baseline_and_directions_*.json` in `OUT_ROOT`.

**Ablauf.** Setup → Drive → Konfiguration → Modell → Stufe B → Kosinus-Heatmap → Unembedding-Tokens → P_freq-vs-Entropie-Streudiagramm → Zusammenfassung.

Hinweise: Die Zellen sind gegen Fehler abgesichert (ein fehlgeschlagener Schritt druckt
den Traceback und das Notebook läuft weiter). Mit `SMOKE = True` läuft alles in
< 3 min auf CPU mit `pythia-14m` (Rauchtest, keine wissenschaftliche Aussage).
Mit `REUSE = True` werden vorhandene Ergebnisse (manifest.json mit `status: ok`)
geladen statt neu gerechnet — so lässt sich nach einem Colab-Abbruch fortsetzen.


In [ ]:
# --- Setup: Repository holen / aktualisieren und installieren -------------------------
# Runs inside Colab (clone into /content/lsne-repo) and inside an existing checkout
# (local Jupyter, tests): a checkout is detected by lsne/__init__.py + pyproject.toml.
import importlib.util
import os
import pathlib
import subprocess
import sys

GIT_URL = os.environ.get("LSNE_GIT_URL", "https://github.com/Erikiss/Language-Switching-.git")
GIT_BRANCH = os.environ.get("LSNE_GIT_BRANCH", "claude/language-switching-non-equilibrium-ejpz0q")
REPO_DIR = os.environ.get("LSNE_REPO_DIR", "/content/lsne-repo")


def _sh(cmd: str) -> int:
    print("$", cmd, flush=True)
    return subprocess.run(cmd, shell=True).returncode


_here = pathlib.Path.cwd().resolve()
_local = next((p for p in [_here, *_here.parents]
               if (p / "lsne" / "__init__.py").exists() and (p / "pyproject.toml").exists()), None)
if _local is not None:
    REPO_DIR = str(_local)
    print("Vorhandener Checkout:", REPO_DIR)
elif pathlib.Path(REPO_DIR, ".git").exists():
    _sh(f"git -C {REPO_DIR} fetch -q --depth 1 origin {GIT_BRANCH} && "
        f"git -C {REPO_DIR} checkout -q {GIT_BRANCH} && git -C {REPO_DIR} pull -q --ff-only origin {GIT_BRANCH}")
else:
    _sh(f"git clone -q --depth 1 --branch {GIT_BRANCH} {GIT_URL} {REPO_DIR}")
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

if _local is None or importlib.util.find_spec("lsne") is None:
    _sh(f"{sys.executable} -m pip install -q -r requirements.txt")
    _sh(f"{sys.executable} -m pip install -q -e .")
import lsne
print("lsne", lsne.__version__, "aus", REPO_DIR, "| Branch", GIT_BRANCH)
print("Git-SHA:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())


In [ ]:
# --- Optional: TNCO (Google Research) ---------------------------------------------------
# TNCO optimises the contraction ORDER of the linearised lattice network (stage E).
# It needs the Boost headers and a C++ build (a few minutes on Colab).  Without TNCO
# lsne.tn_lattice.optimize_path falls back to opt_einsum ("dp" for <= 12 tensors,
# else "greedy"); the contracted response is identical, only the path cost / runtime
# columns of tn_paths differ ("fallback_reason" says why).  Stages A-D never need it.
INSTALL_TNCO = False   # True: bauen (nur für Notebook 04 / Stufe E sinnvoll)

try:
    import tnco
    print("tnco", getattr(tnco, "__version__", "?"), "ist bereits installiert")
except ImportError:
    if INSTALL_TNCO:
        _sh("apt-get install -y -q libboost-dev > /tmp/apt_boost.log 2>&1 || cat /tmp/apt_boost.log | tail -5")
        _sh(f'{sys.executable} -m pip install -q "tnco @ git+https://github.com/google-research/tnco"')
        try:
            import tnco
            print("tnco", getattr(tnco, "__version__", "?"), "installiert")
        except ImportError as e:
            print("TNCO konnte nicht gebaut werden:", repr(e), "-> Fallback opt_einsum (siehe docs/RUNBOOK_COLAB.md)")
    else:
        print("TNCO nicht installiert -> lsne.tn_lattice nutzt opt_einsum (dp/greedy) als Fallback.")


In [ ]:
# --- Google Drive einbinden (außerhalb von Colab wird lokal geschrieben) ---------------
import os

try:
    from google.colab import drive            # type: ignore
    drive.mount("/content/drive")
except Exception as e:                        # not on Colab, or the mount was refused
    print("Kein Colab-Drive:", repr(e).split("(")[0], "-> lokale Ausgabe")

from lsne.io import drive_root

OUT_ROOT = drive_root()                       # Colab: /content/drive/MyDrive/LSNE, sonst $LSNE_RUNS oder ./runs
RUNS_ROOT = os.path.join(OUT_ROOT, "runs")    # Laufverzeichnisse wie in jobs/shards.yaml (--out)
os.makedirs(RUNS_ROOT, exist_ok=True)
if OUT_ROOT.startswith("/content/drive"):
    # share the anchor / unigram caches between sessions (see jobs/shards.yaml defaults.env)
    os.environ.setdefault("LSNE_CACHE", os.path.join(OUT_ROOT, "cache"))
print("OUT_ROOT  =", OUT_ROOT)
print("RUNS_ROOT =", RUNS_ROOT)
print("LSNE_CACHE =", os.environ.get("LSNE_CACHE", "./cache"))


In [ ]:
# --- Konfiguration (hier anpassen) ------------------------------------------------------
NOTEBOOK = "01_baseline_and_directions"
STAGE = "B"
MODEL = "EleutherAI/pythia-160m"   # HF-Modellname (State 2143: pythia-160m)
REVISION = "step98000"             # HF-Revision: step98000 / step99000 (Eingriff), step143000 (SAE)
DTYPE = "float32"                  # "float32" (Standard) | "bfloat16" (eigener Lauf, eigene Null-Dosis-Baseline)
DEVICE = "auto"                    # "auto" | "cuda" | "cpu"
FAMILIES = None                    # z. B. ["F01", "F02"] oder ["DEV01A"]; None = alle Familien (F01-F12, DEV01-DEV04)
SMOKE = False                      # True: Mini-Lauf mit pythia-14m, 1 Familie, 3 Varianten (< 3 min CPU)
SEED = 0
REUSE = True                       # vorhandene Ergebnisse (manifest.json, status ok) laden statt neu rechnen
USE_SAE = False                    # SAE-Feature-Richtungen (EleutherAI/sae-pythia-160m-32k; nur bei step143000 zertifizierbar)
AXIS_NPZ = None                    # optional: axes.npz mit ARI-Achse, gegen die "ent" orthogonalisiert wird
from lsne.config import ModelSpec, RunConfig
from lsne.experiments import common as C

if SMOKE:
    MODEL, REVISION, DTYPE = "EleutherAI/pythia-14m", "step98000", "float32"
    FAMILIES = FAMILIES or ["DEV01"]
spec = ModelSpec(name=MODEL, revision=REVISION, dtype=DTYPE, device=DEVICE)
cfg = RunConfig(model=spec, families=FAMILIES, seed=SEED, out_root=RUNS_ROOT)
limits = C.Limits()
if SMOKE:
    cfg, limits = C.smoke_config(cfg), C.Limits.smoke()
limits.use_sae = bool(USE_SAE) and not SMOKE
limits.progress = True

_fam = f" --families {','.join(FAMILIES)}" if FAMILIES else ""
CLI_COMMAND = (f"python -m lsne.cli run --stage {STAGE} --model {MODEL} --revision {REVISION} --dtype {DTYPE}"
               f" --device {DEVICE} --out {RUNS_ROOT} --seed {SEED}{_fam}{' --smoke' if SMOKE else ''}"
               f"{' --use-sae' if limits.use_sae else ''}")
print("Modell:", spec.tag(), "| Familien:", FAMILIES or "alle", "| SMOKE:", SMOKE)
print("CLI-Äquivalent:\n  " + CLI_COMMAND)


In [ ]:
# --- Modell laden -------------------------------------------------------------------
import torch
from lsne.model import load_model

mb = load_model(spec)
print(f"{spec.tag()}: {mb.n_layers} Blöcke, d_model={mb.d_model}, {mb.n_heads} Heads, "
      f"Vokabular={mb.vocab}, device={mb.device}, dtype={mb.dtype}")
print("Dossier-Eingriffsstelle (verallgemeinert):", C.dossier_site(mb).key,
      "| Tiefen-Injektion (D1b / Stufe E):", C.depth_site(mb).key)


In [ ]:
# --- Runner: Stufen direkt über run() ausführen (Ergebnisse bleiben im Speicher) --------
import contextlib
import json
import time
import traceback

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

from lsne.experiments import STAGES, STAGE_TITLES
from lsne.io import RunDir, load_df, read_manifest

ctx: dict = {}        # shared objects between stages (variants, original, direction bank)
results: dict = {}    # stage -> {table name: DataFrame}
report: dict = {}     # stage -> status / runtime / error
FIG_DIR = os.path.join(OUT_ROOT, "figures", NOTEBOOK)
os.makedirs(FIG_DIR, exist_ok=True)


def run_stage(stage: str, reuse=None, **kw):
    """Run one stage; on failure print the traceback, write error.txt and continue."""
    reuse = REUSE if reuse is None else reuse
    rd = RunDir(cfg.out_root, stage, cfg.model)
    t0 = time.perf_counter()
    if reuse and rd.manifest_path.exists():
        try:
            m = read_manifest(rd)
            if m.get("status") == "ok" and m.get("tables"):
                results[stage] = {name: load_df(rd, name) for name in m["tables"]}
                report[stage] = dict(status="reused", run_dir=str(rd.path), seconds=0.0, error="")
                print(f"Stufe {stage}: vorhandene Ergebnisse geladen aus {rd.path} "
                      f"(REUSE=False erzwingt Neuberechnung)")
                return results[stage]
        except Exception as e:  # unreadable previous run: recompute
            print("Wiederverwendung nicht möglich:", repr(e))
    print(f"Stufe {stage} ({STAGE_TITLES[stage]}) -> {rd.path}", flush=True)
    try:
        results[stage] = STAGES[stage].run(cfg, mb, rd, ctx=ctx, limits=limits, **kw)
        report[stage] = dict(status="ok", run_dir=str(rd.path), seconds=time.perf_counter() - t0, error="")
        print(f"Stufe {stage} ok in {report[stage]['seconds']:.0f} s: {', '.join(sorted(results[stage]))}")
    except Exception as e:
        traceback.print_exc()
        C.write_error(rd, stage, e, cfg, mb)
        report[stage] = dict(status="error", run_dir=str(rd.path), seconds=time.perf_counter() - t0, error=repr(e))
        results[stage] = {}
        print(f"Stufe {stage} FEHLGESCHLAGEN: {e!r} (error.txt in {rd.path})")
    return results[stage]


def table(stage: str, name: str) -> pd.DataFrame:
    """Table of a stage result or an empty DataFrame (keeps the plot cells robust)."""
    df = results.get(stage, {}).get(name)
    return df if isinstance(df, pd.DataFrame) else pd.DataFrame()


def show(df, n: int = 40):
    """display() in Jupyter / Colab, print() elsewhere."""
    if isinstance(df, pd.DataFrame):
        df = df.head(n)
    try:
        display(df)          # noqa: F821 - provided by IPython
    except NameError:
        print(df.to_string() if hasattr(df, "to_string") else df)


def savefig(name: str):
    path = os.path.join(FIG_DIR, f"{name}_{spec.tag().replace(':', '_').replace('@', '_')}.png")
    plt.gcf().savefig(path, dpi=120, bbox_inches="tight")
    print("Abbildung:", path)


@contextlib.contextmanager
def guarded(title: str):
    """Print the traceback of a failing cell section and carry on."""
    print("==", title)
    try:
        yield
    except Exception:
        traceback.print_exc()
        print(f"[{title}] übersprungen (siehe Traceback)")


In [ ]:
# --- Stufe B ausführen ----------------------------------------------------------------
# CLI-Äquivalent:
#   python -m lsne.cli run --stage B --model EleutherAI/pythia-160m --revision step98000 --out <RUNS_ROOT> [--use-sae] [--axis-npz axes.npz]
res_B = run_stage("B", axis_npz=AXIS_NPZ)
print({k: len(v) for k, v in res_B.items()})
if "bank" in ctx:
    bank = ctx["bank"]
    print("Richtungen:", list(bank.keys()))
    print("Übersprungen:", bank.meta.get("skipped"), "| Unigramm-Quelle:", bank.meta.get("unigram"))
    if "sae" in bank.meta:
        print("SAE:", {k: v for k, v in bank.meta["sae"].items() if k not in ("features_D", "features_E")})


In [ ]:
# --- Baseline pro Bedingung -----------------------------------------------------------
with guarded("Baseline log p(Ziel) nach K und C"):
    base = table("B", "baseline")
    std = base[base["kind"] == "standard"] if "kind" in base else base
    g = std.groupby(["K", "C"])[["logp_target", "p_target", "entropy", "p_freq", "target_rank"]].agg(["mean", "std"])
    show(g)
    fig, ax = plt.subplots(figsize=(7, 3.6))
    for Cc, gg in std.groupby("C"):
        m = gg.groupby("K")["logp_target"].agg(["mean", "std"])
        ax.errorbar(m.index, m["mean"], yerr=m["std"].fillna(0), marker="o", capsize=3, label=Cc)
    ax.set_xlabel("K")
    ax.set_ylabel("ln p₀(Ziel)")
    ax.set_title("Reiner Texteffekt des Sprachwechsels auf die Zielvorhersage")
    ax.legend(title="C")
    savefig("baseline_logp_by_K")
    plt.show()


In [ ]:
# --- Kosinus-Heatmap der Richtungen ---------------------------------------------------
with guarded("Kosinus-Heatmap"):
    cos = table("B", "cosines").set_index("name")
    keep = [n for n in cos.index if not n.startswith("rand_") or n in ("rand_00", "rand_01")]
    M = cos.loc[keep, keep].astype(float)
    n = len(keep)
    fig, ax = plt.subplots(figsize=(0.55 * n + 3, 0.55 * n + 2))
    im = ax.imshow(M.to_numpy(), vmin=-1, vmax=1, cmap="RdBu_r")
    ax.set_xticks(range(n))
    ax.set_xticklabels(keep, rotation=90)
    ax.set_yticks(range(n))
    ax.set_yticklabels(keep)
    for i in range(n):
        for j in range(n):
            ax.text(j, i, f"{M.iat[i, j]:+.2f}", ha="center", va="center", fontsize=7)
    meta = table("B", "direction_meta")
    site_key = str(meta["site"].iloc[0]) if len(meta) else C.dossier_site(mb).key
    ax.set_title(f"Kosinus der Richtungen an {site_key} (Zufallsrichtungen: nur rand_00/01 gezeigt)")
    fig.colorbar(im, ax=ax, label="Kosinus", shrink=0.8)
    savefig("direction_cosines")
    plt.show()
    if len(meta):
        cols = [c for c in ["name", "dir_family", "site", "train_acc", "entropy", "source", "fvu", "certified", "selectivity"] if c in meta.columns]
        show(meta[cols], n=60)


In [ ]:
# --- Unembedding: welche Tokens befördert / unterdrückt jede Richtung? ------------------
with guarded("Unembedding-Projektion"):
    un = table("B", "unembed")
    names = [n for n in ["ent", "lang_dom", "lang_probe", "freq", "sae_D0", "sae_E0", "rand_00"] if n in set(un["direction"])]
    for name in names:
        sub = un[un["direction"] == name]
        pro = sub[sub["kind"] == "promoted"].sort_values("rank")["token"].head(15).tolist()
        sup = sub[sub["kind"] == "suppressed"].sort_values("rank")["token"].head(15).tolist()
        print(f"{name:11s} befördert : {pro}")
        print(f"{'':11s} unterdrückt: {sup}")
    # bar chart of the top-10 scores of the two main axes
    fig, axes = plt.subplots(1, min(2, len(names)) or 1, figsize=(11, 3.8), squeeze=False)
    for ax, name in zip(axes[0], [n for n in ["lang_dom", "ent"] if n in names] or names[:2]):
        sub = un[(un["direction"] == name)].sort_values(["kind", "rank"])
        pro = sub[sub["kind"] == "promoted"].head(10)
        sup = sub[sub["kind"] == "suppressed"].head(10)
        y = np.arange(10)
        ax.barh(y + 0.2, pro["score"].to_numpy(), 0.4, color="C3", label="befördert")
        ax.barh(y - 0.2, sup["score"].to_numpy(), 0.4, color="C0", label="unterdrückt")
        ax.set_yticks(y)
        ax.set_yticklabels([f"{repr(p)} / {repr(s)}" for p, s in zip(pro["token"], sup["token"])], fontsize=7)
        ax.invert_yaxis()
        ax.set_title(f"{name}: Unembedding-Score (LN-Gain · W_U)")
        ax.legend(fontsize=7)
    savefig("unembed_top_tokens")
    plt.show()


In [ ]:
# --- P_freq vs. Entropie pro Token, gefärbt nach K -------------------------------------
# Operationalised core of the "frequent-token suppression under uncertainty" idea
# (DESIGN §9): mass on the 100 most frequent Pile tokens against the next-token entropy.
with guarded("P_freq vs. Entropie"):
    tok = table("B", "pfreq_entropy_tokens")
    tok = tok[np.isfinite(tok["p_freq"]) & np.isfinite(tok["entropy"])]
    if len(tok) > 8000:
        tok = tok.sample(8000, random_state=SEED)
    fig, ax = plt.subplots(figsize=(7.5, 5))
    for K, g in sorted(tok.groupby("K")):
        r = np.corrcoef(g["entropy"], g["p_freq"])[0, 1] if len(g) > 2 else float("nan")
        ax.scatter(g["entropy"], g["p_freq"], s=6, alpha=0.45, label=f"K={K} (n={len(g)}, r={r:+.2f})")
    ax.set_xlabel("Entropie der Nächstes-Token-Verteilung [nat]")
    ax.set_ylabel("P_freq: Masse auf den 100 häufigsten Tokens")
    ax.set_title("Unterdrückung häufiger Tokens bei Unsicherheit? (pro Position)")
    ax.legend(fontsize=8)
    savefig("pfreq_vs_entropy")
    plt.show()
    agg = table("B", "pfreq_entropy_by_class")
    if len(agg):
        print("Aggregiert nach (K, C, Tokenklasse, Tail):")
        show(agg.sort_values(["K", "C", "is_tail", "klass"]), n=60)


In [ ]:
# --- Kurze Laufzusammenfassung nach OUT_ROOT schreiben ---------------------------------
with guarded("Zusammenfassung schreiben"):
    import datetime

    summary = dict(
        notebook=NOTEBOOK, stage=STAGE, model=spec.name, revision=spec.revision, dtype=spec.dtype,
        device=str(mb.device), n_layers=mb.n_layers, d_model=mb.d_model, smoke=bool(SMOKE),
        families=FAMILIES, seed=SEED, out_root=OUT_ROOT, runs_root=RUNS_ROOT, cli_command=CLI_COMMAND,
        timestamp=datetime.datetime.now(datetime.timezone.utc).isoformat(),
        report=report,
        tables={st: {n: int(len(df)) for n, df in r.items()} for st, r in results.items()},
        figures=sorted(f for f in os.listdir(FIG_DIR) if f.endswith(".png")),
    )
    path = os.path.join(OUT_ROOT, f"summary_{NOTEBOOK}_{spec.tag().replace(':', '_').replace('@', '_')}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(C.to_jsonable(summary), f, indent=1, ensure_ascii=False)
    print("Zusammenfassung:", path)
    print(json.dumps(C.to_jsonable(summary), indent=1, ensure_ascii=False)[:3000])
